<a href="https://colab.research.google.com/github/jkbacuetes/LarvaSense/blob/main/LarvaSense_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install micromlgen openpyxl

In [15]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, classification_report

from micromlgen import port
from google.colab import files

In [16]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]

print("Uploaded file:", filename)

Saving LarvaSense_Merged_ML_Dataset_Threshold_10000_11000.xlsx to LarvaSense_Merged_ML_Dataset_Threshold_10000_11000.xlsx
Uploaded file: LarvaSense_Merged_ML_Dataset_Threshold_10000_11000.xlsx


In [17]:
df = pd.read_excel(filename, sheet_name="ML_Data")

print("First 5 rows:")
display(df.head())

print("Columns:")
print(df.columns)

print("Dataset size:")
print(df.shape)

First 5 rows:


,position,sample_id,v610,v680,v730,v760,v810,v860,label,label_text,data_origin,threshold_rule
0,Side 2,I-SIDE2-T003,10000.0,9131.8,10000.0,9808.5,9044.6,8417.9,1,INFESTED / UNHEALTHY,Generated_Threshold,<= 10000
1,Top,U-TOP-T004,13262.5,12484.8,13399.0,12416.2,11469.5,11340.4,0,UNINFESTED / HEALTHY,Generated_Threshold,>= 11000
2,Side 3,U-SIDE3-T024,13568.8,13096.5,13821.3,13699.4,12940.5,12201.9,0,UNINFESTED / HEALTHY,Generated_Threshold,>= 11000
3,Bottom,I-BOTTOM-T011,8745.8,8567.9,9092.0,8399.0,7885.7,8064.2,1,INFESTED / UNHEALTHY,Generated_Threshold,<= 10000
4,Side 2,U-SIDE2-T023,13930.4,13555.8,14844.8,13895.0,12884.0,12515.3,0,UNINFESTED / HEALTHY,Generated_Threshold,>= 11000


Columns:
Index(['position', 'sample_id', 'v610', 'v680', 'v730', 'v760', 'v810', 'v860',
       'label', 'label_text', 'data_origin', 'threshold_rule'],
      dtype='object')
Dataset size:
(300, 12)


In [18]:
print("Label count:")
print(df['label'].value_counts())

print("\nLabel text count:")
print(df['label_text'].value_counts())

Label count:
label
1    150
0    150
Name: count, dtype: int64

Label text count:
label_text
INFESTED / UNHEALTHY    150
UNINFESTED / HEALTHY    150
Name: count, dtype: int64


In [19]:
features = ['v610', 'v680', 'v730', 'v760', 'v810', 'v860']

X = df[features].values.astype(float)
y = df['label'].values.astype(int)

print("Sensor data shape:", X.shape)
print("Label data shape:", y.shape)

print("\nLabel count:")
print(pd.Series(y).value_counts())

Sensor data shape: (300, 6)
Label data shape: (300,)

Label count:
1    150
0    150
Name: count, dtype: int64


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 210
Testing samples: 90


In [21]:
start_time = time.time()

model = SVC(
    kernel='linear',
    C=1.0,
    gamma=0.001
)

model.fit(X_train, y_train)

end_time = time.time()

print("SVM model training finished!")
print("Training time:", end_time - start_time, "seconds")
print("Number of support vectors:", model.support_vectors_.shape[0])

SVM model training finished!
Training time: 0.006385087966918945 seconds
Number of support vectors: 2


In [22]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=1)

print("Accuracy:", accuracy * 100, "%")
print("Precision for INFESTED:", precision * 100, "%")

Accuracy: 100.0 %
Precision for INFESTED: 100.0 %


In [23]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Uninfested / Healthy', 'Infested / Unhealthy']
))

Confusion Matrix:
[[45  0]
 [ 0 45]]

Classification Report:
                      precision    recall  f1-score   support

Uninfested / Healthy       1.00      1.00      1.00        45
Infested / Unhealthy       1.00      1.00      1.00        45

            accuracy                           1.00        90
           macro avg       1.00      1.00      1.00        90
        weighted avg       1.00      1.00      1.00        90



In [24]:
infested_sample = df[df['label'] == 1][features].iloc[0].values.reshape(1, -1)
uninfested_sample = df[df['label'] == 0][features].iloc[0].values.reshape(1, -1)

infested_prediction = model.predict(infested_sample)[0]
uninfested_prediction = model.predict(uninfested_sample)[0]

print("Known INFESTED sample prediction:", infested_prediction)
if infested_prediction == 1:
    print("Result: INFESTED")
else:
    print("Result: UNINFESTED")

print("\nKnown UNINFESTED sample prediction:", uninfested_prediction)
if uninfested_prediction == 1:
    print("Result: INFESTED")
else:
    print("Result: UNINFESTED")

Known INFESTED sample prediction: 1
Result: INFESTED

Known UNINFESTED sample prediction: 0
Result: UNINFESTED


In [25]:
# CELL 12 — Test actual rows from your latest spreadsheet

features = ['v610', 'v680', 'v730', 'v760', 'v810', 'v860']

# Get one known infested row from your spreadsheet
infested_row = df[df['label'] == 1].iloc[0]

# Get one known uninfested row from your spreadsheet
uninfested_row = df[df['label'] == 0].iloc[0]

# Prepare values
infested_sample = infested_row[features].values.astype(float).reshape(1, -1)
uninfested_sample = uninfested_row[features].values.astype(float).reshape(1, -1)

# Predict
infested_prediction = model.predict(infested_sample)[0]
uninfested_prediction = model.predict(uninfested_sample)[0]

# Print actual spreadsheet values
print("KNOWN INFESTED SAMPLE FROM SPREADSHEET")
print("Sample ID:", infested_row['sample_id'])
print("Position:", infested_row['position'])
print("Values:", infested_sample)
print("Total Signal:", infested_sample.sum())
print("Model Prediction:", infested_prediction)

if infested_prediction == 1:
    print("Result: INFESTED ✅")
else:
    print("Result: UNINFESTED ❌")

print("\nKNOWN UNINFESTED SAMPLE FROM SPREADSHEET")
print("Sample ID:", uninfested_row['sample_id'])
print("Position:", uninfested_row['position'])
print("Values:", uninfested_sample)
print("Total Signal:", uninfested_sample.sum())
print("Model Prediction:", uninfested_prediction)

if uninfested_prediction == 0:
    print("Result: UNINFESTED ✅")
else:
    print("Result: INFESTED ❌")

KNOWN INFESTED SAMPLE FROM SPREADSHEET
Sample ID: I-SIDE2-T003
Position: Side 2
Values: [[10000.   9131.8 10000.   9808.5  9044.6  8417.9]]
Total Signal: 56402.8
Model Prediction: 1
Result: INFESTED ✅

KNOWN UNINFESTED SAMPLE FROM SPREADSHEET
Sample ID: U-TOP-T004
Position: Top
Values: [[13262.5 12484.8 13399.  12416.2 11469.5 11340.4]]
Total Signal: 74372.4
Model Prediction: 0
Result: UNINFESTED ✅


In [26]:
# CELL 12B — Check if the model predicts the whole spreadsheet correctly

all_predictions = model.predict(X)

df_check = df.copy()
df_check['prediction'] = all_predictions
df_check['correct'] = df_check['label'] == df_check['prediction']
df_check['total_signal'] = df_check[features].sum(axis=1)

print("Total rows:", len(df_check))
print("Correct predictions:", df_check['correct'].sum())
print("Wrong predictions:", (~df_check['correct']).sum())
print("Accuracy on full spreadsheet:", df_check['correct'].mean() * 100, "%")

print("\nWrong predictions:")
display(df_check[df_check['correct'] == False][[
    'sample_id', 'position', 'label', 'label_text',
    'prediction', 'total_signal',
    'v610', 'v680', 'v730', 'v760', 'v810', 'v860'
]])

Total rows: 300
Correct predictions: 300
Wrong predictions: 0
Accuracy on full spreadsheet: 100.0 %

Wrong predictions:


,sample_id,position,label,label_text,prediction,total_signal,v610,v680,v730,v760,v810,v860


In [27]:
# CELL 13 — Convert trained SVM model to model.h

c_code = port(model, classmap={
    0: 'UNINFESTED',
    1: 'INFESTED'
})

with open('model.h', 'w') as file:
    file.write(c_code)

print("New model.h file created successfully!")

New model.h file created successfully!


In [28]:
files.download('model.h')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>